# LBSN Preprocess

In [9]:
import os
import os.path as osp
import pickle
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Tuple, Dict, Any, List

import pandas as pd
from sklearn.preprocessing import LabelEncoder


### Config

In [2]:
@dataclass
class PreprocessConfig:
    dataset_root: str  # path to one dataset folder
    data: str      # dataset name, e.g., "NYC"

    # Filtering thresholds
    min_poi_freq: int = 10
    min_user_freq: int = 10

    # Session split (minutes). New session if gap > interval.
    session_time_interval_min: int = 60 * 24

    # Global split ratios by time (not per-user)
    train_ratio: float = 0.8
    val_ratio: float = 0.1
    # test_ratio = 1 - train - val

    # Remove isolated points with both-side gaps > 24h
    remove_isolated_24h: bool = True

    # Ignore trajectories with only 1 check-in
    ignore_singleton_trajectories: bool = True

    # Label encoding: padding id = 0, all real ids shifted by +1
    padding_id: int = 0


cfg = PreprocessConfig(
    dataset_root="/data/home/dswang/new_disk/dswang/code/DPO4POI/new_code/data",
    data="NYC",
)


RAW_PATH = osp.join(cfg.dataset_root, cfg.data, cfg.data+".csv")
OUT_DIR = osp.join(cfg.dataset_root, cfg.data)

### Helpers (simple & explicit)

In [ ]:
def read_format(file_path: str) -> pd.DataFrame:

    df = pd.read_csv(file_path)
    # UId,PId,Category,Latitude,Longitude,Time,Weekday

    df["Time"] = pd.to_datetime(df["Time"], format="%Y-%m-%d %H:%M:%S", errors="raise")
    df = df.sort_values(["UId", "Time"]).reset_index(drop=True)
    return df


def filter_low_frequency(df: pd.DataFrame, min_poi_freq: int, min_user_freq: int) -> pd.DataFrame:
    """Drop low-frequency POIs and users."""
    poi_count = df.groupby('PId')['UId'].count()
    keep_pois = set(poi_count[poi_count > min_poi_freq].index)
    df = df[df['PId'].isin(keep_pois)]

    user_count = df.groupby('UId')['PId'].count()
    keep_users = set(user_count[user_count > min_user_freq].index)
    df = df[df['UId'].isin(keep_users)]

    return df.sort_values(by=['UId', 'Time'], ascending=True).reset_index(drop=True)


def split_by_global_time(df: pd.DataFrame, train_ratio: float, val_ratio: float) -> pd.DataFrame:
    """Split by global time order: earliest -> train, middle -> val, latest -> test."""
    df = df.sort_values("Time").reset_index(drop=True)
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    df['SplitTag'] = 'train'
    df.loc[train_end:val_end-1, 'SplitTag'] = 'validation'
    df.loc[val_end:, 'SplitTag'] = 'test'
    
    # re-sort back to per-user time order
    return df.sort_values(["UId", "Time"]).reset_index(drop=True)


def remove_isolated_checkins_24h(df: pd.DataFrame) -> pd.DataFrame:
    """Remove check-ins whose gaps to both previous and next check-in (same user) are > 24h."""
    df = df.sort_values(by=['UId', 'Time'], ascending=True).reset_index(drop=True)

    prev_t = df.groupby("UId")["Time"].shift(1)
    next_t = df.groupby("UId")["Time"].shift(-1)

    gap_prev = (df["Time"] - prev_t).dt.total_seconds().fillna(float("inf"))
    gap_next = (next_t - df["Time"]).dt.total_seconds().fillna(float("inf"))

    isolated = (gap_prev > 86400) & (gap_next > 86400)
    return df[~isolated].reset_index(drop=True)


def build_pseudo_sessions(df: pd.DataFrame, session_time_interval_min: int) -> pd.DataFrame:
    """Create pseudo_session_trajectory_id by splitting each user's timeline with a time-gap threshold."""
    df = df.sort_values(by=['UId', 'Time'], ascending=True).reset_index(drop=True)
    # time diff per user
    diffs = df.groupby('UId')['Time'].diff()
    diffs_min = diffs.dt.total_seconds() / 60.0
    df['time_interval'] = diffs_min

    # new session if: first record of user OR diff > threshold
    new_session = diffs.isna() | (diffs_min > session_time_interval_min)
    # Assign a global session id by cumulative sum of new_session flags
    df['pseudo_session_trajectory_id'] = new_session.cumsum().astype(int) - 1
    return df


def ignore_singleton_sessions(df: pd.DataFrame) -> pd.DataFrame:
    """Mark sessions (trajectories) with only 1 check-in as ignored."""
    counts = df.groupby('pseudo_session_trajectory_id')['Time'].transform('count')
    df.loc[counts == 1, 'SplitTag'] = 'ignore'
    return df


def id_encode_padding0(fit_df: pd.DataFrame, encode_df: pd.DataFrame, column: str) -> Tuple[LabelEncoder, int]:
    """Label encode with padding id = 0; real ids shifted by +1."""
    le = LabelEncoder().fit(fit_df[column].tolist())
    pad = 0
    classes = set(le.classes_)
    encode_df[column] = [le.transform([v])[0] + 1 if v in classes else pad for v in encode_df[column].tolist()]
    return le, pad


def remove_unseen_user_poi(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    train = df[df['SplitTag'] == 'train']
    val = df[df['SplitTag'] == 'validation']
    test = df[df['SplitTag'] == 'test']

    train_users = set(train['UId'])
    train_pois = set(train['PId'])

    val = val[val['UId'].isin(train_users) & val['PId'].isin(train_pois)].reset_index(drop=True)
    test = test[test['UId'].isin(train_users) & test['PId'].isin(train_pois)].reset_index(drop=True)

    sample = pd.concat(
        [train.reset_index(drop=True), val, test],
        axis=0
    ).sort_values(['UId', 'Time']).reset_index(drop=True)

    return {
        'sample': sample,
        'train_sample': train.reset_index(drop=True),
        'validate_sample': val,
        'test_sample': test
    }


def add_sequence_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add sequence_id: the index of this check-in in the user's full check-in sequence (0-based).
    Requires df to have columns: UId, Time (Time must be datetime-like).
    """
    df = df.sort_values(by=['UId', 'Time'], ascending=True).reset_index(drop=True)

    df["sequence_id"] = df.groupby("UId").cumcount().astype("int64")
    return df

def build_id_map_from_train(train_series: pd.Series):
    uniq = pd.unique(train_series)
    return {v: i+1 for i, v in enumerate(uniq)}, 0

def apply_map_inplace(df: pd.DataFrame, col: str, mapping: dict, pad: int = 0):
    df[col] = df[col].map(mapping).fillna(pad).astype("int64")

### Run preprocessing

In [7]:
# Read
df = read_format(RAW_PATH)
print('After read:', df.shape, 'users', df.UId.nunique(), 'pois', df.PId.nunique())

# Filter low frequency
df = filter_low_frequency(df, cfg.min_poi_freq, cfg.min_user_freq)
print('After filter:', df.shape, 'users', df.UId.nunique(), 'pois', df.PId.nunique())

# 3) Split train/val/test by global time
df = split_by_global_time(df, cfg.train_ratio, cfg.val_ratio)
print(df['SplitTag'].value_counts())

# Optional: remove isolated check-ins
if cfg.remove_isolated_24h:
    df = remove_isolated_checkins_24h(df)
    print('After removing isolated 24h:', df.shape)

# Build pseudo sessions
df = build_pseudo_sessions(df, cfg.session_time_interval_min)
print('sessions:', df['pseudo_session_trajectory_id'].nunique())

# Assign check_in id (global time rank)
df = df.sort_values('Time', ascending=True).reset_index(drop=True)
df['check_ins_id'] = df['Time'].rank(ascending=True, method='first').astype('int64') - 1
df = df.sort_values(['UId', 'Time'], ascending=True).reset_index(drop=True)

# Ignore singleton sessions
if cfg.ignore_singleton_trajectories:
    df = ignore_singleton_sessions(df)
    print('ignored:', (df['SplitTag']=='ignore').sum())

# Remove unseen users/pois from val/test
result = remove_unseen_user_poi(df)
print('sample:', result['sample'].shape,
      'train:', result['train_sample'].shape,
      'val:', result['validate_sample'].shape,
      'test:', result['test_sample'].shape)


# Add sequence_id
final_df = add_sequence_id(result['sample'])


# Label encoding (fit on train only)
train_df = final_df[final_df['SplitTag'] == 'train']
uid_map, uid_pad = build_id_map_from_train(train_df['UId'])
pid_map, pid_pad = build_id_map_from_train(train_df['PId'])
apply_map_inplace(final_df, 'UId', uid_map, uid_pad)
apply_map_inplace(final_df, 'PId', pid_map, pid_pad)
cat_map, cat_pad = build_id_map_from_train(train_df['Category'])
final_df['CategoryId'] = final_df['Category'].map(cat_map).fillna(cat_pad).astype('int64')
final_df['WeekdayId'] = final_df['Time'].dt.weekday.astype('int64')
final_df['Hour'] = final_df['Time'].dt.hour.astype('int64')

# change column order
final_df = final_df[['UId', 'PId', 'CategoryId', 'Category', 'Latitude', 'Longitude', 
                     'Weekday', 'WeekdayId', 'Time','Hour', 'SplitTag', 'pseudo_session_trajectory_id',
                     'sequence_id']]

# Save CSV outputs
sample_path = osp.join(OUT_DIR, 'sample.csv')
train_path = osp.join(OUT_DIR, 'train_sample.csv')
val_path = osp.join(OUT_DIR, 'validate_sample.csv')
test_path = osp.join(OUT_DIR, 'test_sample.csv')

final_df.to_csv(sample_path, index=False)
final_df[final_df['SplitTag'] == 'train'].to_csv(train_path, index=False)
final_df[final_df['SplitTag'] == 'validation'].to_csv(val_path, index=False)
final_df[final_df['SplitTag'] == 'test'].to_csv(test_path, index=False)

print('Saved:')
print(' ', sample_path)
print(' ', train_path)
print(' ', val_path)
print(' ', test_path)


After read: (227428, 7) users 1083 pois 38333
After filter: (142958, 7) users 1082 pois 4638
SplitTag
train         114366
validation     14296
test           14296
Name: count, dtype: int64
After removing isolated 24h: (119713, 8)
sessions: 23107
ignored: 0
sample: (118320, 11) train: (97730, 11) val: (10493, 11) test: (10097, 11)
Saved:
  /data/home/dswang/new_disk/dswang/code/DPO4POI/new_code/data/NYC/sample.csv
  /data/home/dswang/new_disk/dswang/code/DPO4POI/new_code/data/NYC/train_sample.csv
  /data/home/dswang/new_disk/dswang/code/DPO4POI/new_code/data/NYC/validate_sample.csv
  /data/home/dswang/new_disk/dswang/code/DPO4POI/new_code/data/NYC/test_sample.csv


In [8]:
print('SplitTag counts in saved sample:')
print(pd.read_csv(sample_path)['SplitTag'].value_counts())

print('\nExample rows:')
pd.read_csv(train_path).head(3)

SplitTag counts in saved sample:
SplitTag
train         97730
validation    10493
test          10097
Name: count, dtype: int64

Example rows:


,UId,PId,CategoryId,Category,Latitude,Longitude,Weekday,WeekdayId,Time,Hour,SplitTag,pseudo_session_trajectory_id,sequence_id
0,1,1,1,General Entertainment,40.739398,-73.993210,Sunday,6,2012-04-08 14:20:29,14,train,0,0
1,1,2,2,Breakfast Spot,40.719929,-74.008532,Monday,0,2012-04-09 12:20:52,12,train,0,1
2,1,3,3,Bar,40.760667,-73.994948,Friday,4,2012-04-13 21:11:20,21,train,1,2


### Check-in Sequnce

In [12]:
import pandas as pd
from collections import defaultdict

def build_user_sequence_dict(sample_df: pd.DataFrame) -> dict:
    """
    Build user sequence dictionary from sample dataframe.

    Returns:
      {
        UId: {
          "PIds": [...],
          "CategoryIds": [...],
          "Categorys": [...],
          "Latitudes": [...],
          "Longitudes": [...],
          "Weekdays": [...],
          "WeekdayIds": [...],
          "Times": [...],
          "Hours": [...]
        }
      }
    """
    user_seq = {}
    sample_df = sample_df.sort_values(
        by=["UId", "Time"], ascending=True
    )

    for uid, g in sample_df.groupby("UId"):
        user_seq[int(uid)] = {
            "PIds": g["PId"].astype(int).tolist(),
            "CategoryIds": g["CategoryId"].astype(int).tolist(),
            "Categorys": g["Category"].astype(str).tolist(),
            "Latitudes": g["Latitude"].astype(float).tolist(),
            "Longitudes": g["Longitude"].astype(float).tolist(),
            "Weekdays": g["Weekday"].astype(str).tolist(),
            "WeekdayIds": g["WeekdayId"].astype(int).tolist(),
            "Times": g["Time"].astype(str).tolist(),
            "Hours": g["Hour"].astype(int).tolist(),
        }

    return user_seq

def build_session_index_df(
    split_df: pd.DataFrame,
    min_start_seq: int = 10,
    sequence_length: int = 50,
) -> pd.DataFrame:
    """
    Build session-level index dataframe:
    columns = [UId, session_id, start_seq, end_seq]

    Requirements for split_df:
      - columns: UId, pseudo_session_trajectory_id, sequence_id
      - sequence_id: user-wise, 0-based, continuous
    """
    records: List[dict] = []
    split_df = split_df.sort_values(["UId", "sequence_id"]).reset_index(drop=True)

    for session_id, g in split_df.groupby("pseudo_session_trajectory_id"):
        uids = g["UId"].unique()
        if len(uids) != 1:
            continue

        uid = int(uids[0])
        start_seq = int(g["sequence_id"].min())
        end_seq = int(g["sequence_id"].max())

        if start_seq < min_start_seq:
            continue
        
        start_seq = max(0, end_seq - sequence_length + 1)

        records.append({
            "UId": uid,
            "session_id": int(session_id),
            "start_seq": start_seq,
            "end_seq": end_seq,
        })

    return pd.DataFrame(records)


In [ ]:
import orjson
sample_df = pd.read_csv(
    osp.join(OUT_DIR, "sample.csv"),
    parse_dates=["Time"]
)
user_sequence_dict = build_user_sequence_dict(sample_df)

with open(osp.join(OUT_DIR, "user_sequences.pkl"), "wb") as f:
    pickle.dump(user_sequence_dict, f)

train_df = pd.read_csv(osp.join(OUT_DIR, "train_sample.csv"))
val_df   = pd.read_csv(osp.join(OUT_DIR, "validate_sample.csv"))
test_df  = pd.read_csv(osp.join(OUT_DIR, "test_sample.csv"))


train_idx = build_session_index_df(train_df, min_start_seq=20, sequence_length=50)
val_idx   = build_session_index_df(val_df, sequence_length=50)
test_idx  = build_session_index_df(test_df, sequence_length=50)

print("train / val / test samples:",
      len(train_idx), len(val_idx), len(test_idx))


train_idx.to_csv(osp.join(OUT_DIR, "train_session.csv"), index=False)
val_idx.to_csv(osp.join(OUT_DIR, "val_session.csv"), index=False)
test_idx.to_csv(osp.join(OUT_DIR, "test_session.csv"), index=False)

### POI information

In [ ]:
def poi_info(df):

    df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
    poi_info_data = []

    for pid, group in df.groupby('PId'):
        row0 = group.iloc[0]
        category = row0['Category']
        latitude = row0['Latitude']
        longitude = row0['Longitude']

        hours = group['Time'].dt.hour
        hour_counts = hours.value_counts().to_dict()  # {hour: count}

        count_threshold = 1
        filtered_hour_counts = {int(h): int(c) for h, c in hour_counts.items() if c >= count_threshold}

        sorted_hour_counts = dict(
            sorted(filtered_hour_counts.items(), key=lambda x: x[1], reverse=True)
        )

        poi_info_data.append({
            'pid': pid,
            'category': category,
            'latitude': latitude,
            'longitude': longitude,
            'visit_time_and_count': sorted_hour_counts
        })

    poi_info_df = pd.DataFrame(poi_info_data)
    output_path = f'{OUT_DIR}/poi_info.csv'
    poi_info_df.to_csv(output_path, index=False)

    print(f"{output_path}，{len(poi_info_df)} POI")


sample_df = pd.read_csv(
    osp.join(OUT_DIR, "sample.csv"),
    parse_dates=["Time"]
)
poi_info(sample_df)

/data/home/dswang/new_disk/dswang/code/DPO4POI/new_code/data/NYC/poi_info.csv，4576 POI


### LLM data

In [ ]:
import os
import os.path as osp
import json
import pickle
import ast
import pandas as pd
from typing import Dict, Any, List, Tuple, Optional


INSTRUCTION = (
    "Here is a record of a user's POI accesses, your task is based on the history to predict the POI that the user is likely to access at the specified time."
)

LETTERS = "abcdefghijklmnopqrstuvwxyz"


def sid_list_to_tokens(sid_list: List[int]) -> str:
    """[14, 52, 5, 0] -> <a_14><b_52><c_5><d_0>"""
    toks = []
    for i, v in enumerate(sid_list):
        if i >= len(LETTERS):
            toks.append(f"<x{i}_{int(v)}>")
        else:
            toks.append(f"<{LETTERS[i]}_{int(v)}>")
    return "".join(toks)


def load_pid_to_tokens(semitic_csv_path: str) -> Dict[int, str]:
    """
    semitic_codes.csv:
      pid,sid
      1,"[14, 52, 5, 0]"
    -> {1: "<a_14><b_52><c_5><d_0>", ...}
    """
    df = pd.read_csv(semitic_csv_path)
    pid2tok: Dict[int, str] = {}

    for _, row in df.iterrows():
        pid = int(row["pid"])
        sid_raw = row["sid"]
        try:
            sid_list = ast.literal_eval(sid_raw) if isinstance(sid_raw, str) else list(sid_raw)
            if not isinstance(sid_list, list):
                sid_list = list(sid_list)
            pid2tok[pid] = sid_list_to_tokens([int(x) for x in sid_list])
        except Exception:
            pid2tok[pid] = "<unk>"

    return pid2tok


def load_user_sequences(pkl_path: str) -> Dict[Any, Dict[str, List]]:
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)
    return data


def get_user_seq(user_sequences: Dict[Any, Dict[str, List]], uid: int) -> Optional[Dict[str, List]]:
    """
    兼容 uid key 是 int 或 str 的情况
    """
    if uid in user_sequences:
        return user_sequences[uid]
    suid = str(uid)
    if suid in user_sequences:
        return user_sequences[suid]
    return None


def fmt_time(t: Any) -> str:
    if pd.isna(t):
        return ""
    if hasattr(t, "strftime"):
        return t.strftime("%Y-%m-%d %H:%M")
    s = str(t)
    try:
        ts = pd.to_datetime(s)
        return ts.strftime("%Y-%m-%d %H:%M")
    except Exception:
        return s[:16].replace("T", " ")


def build_one_example(
    uid_new: int,
    times: List[Any],
    pids: List[int],
    pid2tok: Dict[int, str],
    start_seq: int,
    end_seq: int
) -> Optional[Dict[str, str]]:

    if end_seq <= start_seq:
        return None
    if end_seq >= len(pids) or end_seq >= len(times):
        return None

    hist_pids = pids[start_seq:end_seq]
    hist_times = times[start_seq:end_seq]

    target_pid = int(pids[end_seq])
    target_time = times[end_seq]

    if len(hist_pids) == 0:
        return None

    hist_parts = []
    for tt, pid in zip(hist_times, hist_pids):
        tok = pid2tok.get(int(pid), "<unk>")
        hist_parts.append(f"{fmt_time(tt)} visited {tok}")

    history_str = ", ".join(hist_parts)
    input_text = (
        f"User_{uid_new} checkin history: {history_str}.\n"
        f"When {fmt_time(target_time)} user_{uid_new} is likely to visit:"
    )

    output_text = pid2tok.get(target_pid, "<unk>")

    return {
        "instruction": INSTRUCTION,
        "input": input_text,
        "output": output_text
    }


def build_llm_json_from_sessions(
    session_csv_path: str,
    user_sequences: Dict[Any, Dict[str, List]],
    pid2tok: Dict[int, str],
    out_json_path: str
) -> None:
    sess_df = pd.read_csv(session_csv_path)

    unique_uids = sess_df["UId"].dropna().astype(int).unique().tolist()
    uid_map = {int(uid): i for i, uid in enumerate(unique_uids)}

    examples: List[Dict[str, str]] = []

    for _, row in sess_df.iterrows():
        uid = int(row["UId"])
        start_seq = int(row["start_seq"])
        end_seq = int(row["end_seq"])

        user_seq = get_user_seq(user_sequences, uid)
        if user_seq is None:
            continue

        if "PIds" not in user_seq or "Times" not in user_seq:
            continue

        pids = user_seq["PIds"]
        times = user_seq["Times"]

        ex = build_one_example(
            uid_new=uid_map[uid],
            times=times,
            pids=pids,
            pid2tok=pid2tok,
            start_seq=start_seq,
            end_seq=end_seq
        )
        if ex is not None:
            examples.append(ex)

    os.makedirs(osp.dirname(out_json_path), exist_ok=True)
    with open(out_json_path, "w", encoding="utf-8") as f:
        json.dump(examples, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(examples)} examples -> {out_json_path}")


if __name__ == "__main__":

    user_sequences_pkl = osp.join(OUT_DIR, "user_sequences.pkl")
    semitic_codes_csv = osp.join(OUT_DIR, "semitic_codes.csv")

    train_session_csv = osp.join(OUT_DIR, "train_session.csv")
    val_session_csv = osp.join(OUT_DIR, "val_session.csv")
    test_session_csv = osp.join(OUT_DIR, "test_session.csv")

    train_out = osp.join(OUT_DIR, "train_llm.json")
    val_out = osp.join(OUT_DIR, "val_llm.json")
    test_out = osp.join(OUT_DIR, "test_llm.json")

    user_sequences = load_user_sequences(user_sequences_pkl)
    pid2tok = load_pid_to_tokens(semitic_codes_csv)

    build_llm_json_from_sessions(train_session_csv, user_sequences, pid2tok, train_out)
    build_llm_json_from_sessions(val_session_csv, user_sequences, pid2tok, val_out)
    build_llm_json_from_sessions(test_session_csv, user_sequences, pid2tok, test_out)

Saved 12409 examples -> /data/home/dswang/new_disk/dswang/code/DPO4POI/new_code/data/NYC/train_llm.json
Saved 2448 examples -> /data/home/dswang/new_disk/dswang/code/DPO4POI/new_code/data/NYC/val_llm.json
Saved 2360 examples -> /data/home/dswang/new_disk/dswang/code/DPO4POI/new_code/data/NYC/test_llm.json
